In [110]:
from pathlib import Path
import sys

try:
    # .py 脚本
    current_dir = Path(__file__).resolve().parent
except NameError:
    # .ipynb
    current_dir = Path.cwd()

project_root = current_dir.parent
sys.path.append(str(project_root))

import logging
from config.log_config import setup_logging

setup_logging()
logger = logging.getLogger(__name__)

In [111]:
from rich.console import Console


console = Console()


def showtext(text):
    try:
        console.print(text)
    except Exception as e:
        print(f"Console print failed: {e}\n")
        print(text)

In [112]:
todos = []  # 需要完成的任务
completed = []  # 标记任务完成状态

In [113]:
def get_todo_report():
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:  # 已经完成
            result += f"待完成 #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"待完成 #{index + 1}: {todo}\n"
    showtext(result)
    return result

In [114]:
# 扩展 todos, completed(默认 False)
def create_todos(descriptions: list[str]):
    todos.extend(descriptions)
    completed.extend([False] * len(todos))  # 默认状态全为 false
    return get_todo_report()

In [115]:
def mark_complete(index: int, complete_notes: str):  # index 是 complete_notes 的 index
    i = index - 1  # index 起始编号为 1
    if 0 <= i < len(todos):
        completed[i] = True  # 更新状态
        showtext(complete_notes)
        return get_todo_report()
    else:
        return f"索引{index}没有对应的待完成事项"

In [116]:
todos.clear()
completed.clear()

In [117]:
create_todos(["买杂货", "完成其它实验", "吃苹果"])

待完成 #1: 买杂货
待完成 #2: 完成其它实验
待完成 #3: 吃苹果

'待完成 #1: 买杂货\n待完成 #2: 完成其它实验\n待完成 #3: 吃苹果\n'

In [118]:
mark_complete(1, "买")

买

待完成 #1: 买杂货
待完成 #2: 完成其它实验
待完成 #3: 吃苹果

'待完成 #1: [green][strike]买杂货[/strike][/green]\n待完成 #2: 完成其它实验\n待完成 #3: 吃苹果\n'

In [119]:
todos.clear()
completed.clear()

In [120]:
# function 元数据
# "description" 是对函数的描述
# "properties" 下的"descriptions" 是参数名称
# "title" 的作用类似 lable, 给参数加一个标签
create_todos_json = {
    "name": "create_todos",
    "description": "添加新待办事项并返回包含所有待办状态的完整报告",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                "type": "array",
                "items": {"type": "string"},
                "title": "Descriptions",
            }
        },
        "required": ["descriptions"],
        "additionalProperties": False,
    },
}

In [121]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "将编号为 index 的代办项标记为 True 并返回包含所有待办状态的完整报告",
    "parameters": {
        "type": "object",
        "properties": {
            "index": {"type": "integer", "title": "Index"},
            "complete_notes": {"type": "string", "title": "CompleteNotes"},
        },
        "required": ["index", "complete_notes"],
        "additionalProperties": False,
    },
}

In [122]:
tools = [
    {"type": "function", "function": create_todos_json},
    {"type": "function", "function": mark_complete_json},
]

In [123]:
from openai.types.chat import (
    ParsedChatCompletion,
    ChatCompletionMessageToolCallUnion,
    ChatCompletionMessageFunctionToolCall,
    ChatCompletionMessage,
)
import json

In [124]:
def handle_tool_json(tool_calls: list[ChatCompletionMessageToolCallUnion]):
    results = []
    for tool_call in tool_calls:
        if (
            isinstance(tool_call, ChatCompletionMessageFunctionToolCall)
            and tool_call.function is not None
        ):
            tool_name = tool_call.function.name
            parameters = json.loads(tool_call.function.arguments)

            tool = globals().get(
                tool_name
            )  # globals 包括函数, 类, 全局变量的字符串名称
            todo_result = tool(**parameters) if tool else {}
            results.append(
                {
                    "role": "tool",
                    "content": json.dumps(todo_result),
                    "tool_call_id": tool_call.id,
                }
            )
    return results

In [125]:
import os
qwen_api_key = os.getenv("DASHSCOPE_API_KEY")

if qwen_api_key:
    logger.info(f"qwen_api_key: {qwen_api_key[:8]}")
else:
    logger.error("ai api key is invalid")

2026-06-26 14:30:50,225 [INFO] __main__: qwen_api_key: sk-37893


In [126]:
from openai import OpenAI


qwen_client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

In [ ]:
def loop(messages: list[dict[str, str]]):
    done = False
    while not done:
        response = qwen_client.chat.completions.create(
            model="qwen3.7-plus",
            messages=messages,  # type: ignore
            tools=tools,  # type: ignore
            reasoning_effort="none",
        )

        re_choice = response.choices[0]
        finish_reason = re_choice.finish_reason
        if finish_reason == "tool_calls":
            tool_call_message = re_choice.message
            tool_calls = tool_call_message.tool_calls
            if tool_calls:
                tool_call_results = handle_tool_json(tool_calls)
                messages.append(tool_call_message.model_dump(exclude_none=True))
                messages.extend(tool_call_results)
        else:
            done = True
    if response:  # type: ignore
        showtext(response.choices[0].message.content) # 模型的回答

In [128]:
system_message = """你被赋予了一个需要解决的问题。你需要使用你的待办事项（todo）工具来规划一个步骤列表，然后依次执行每个步骤。
现在，请使用待办事项列表工具创建一个计划，执行这些步骤，并回复解决方案。如果问题中没有提供某个数值，请在计划中包含一个步骤来给出合理的估计。
请使用 Rich 控制台标记（Rich console markup）提供你的解决方案，不要使用代码块（code blocks）。
不要向用户提问或要求澄清；在使用工具后，只需直接回答结果。"""

user_message = """一列火车在下午 2:00 离开波士顿，以每小时 60 英里的速度行驶。
另一列火车在下午 3:00 离开纽约，以每小时 80 英里的速度向波士顿方向行驶。它们何时相遇？"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message},
]

loop(messages)

待完成 #1: 估计波士顿到纽约之间的距离
待完成 #2: 计算第一列火车在下午2:00到3:00之间行驶的距离
待完成 #3: 计算第二列火车出发时两列火车之间的剩余距离
待完成 #4: 计算两列火车的相对速度（相向而行）
待完成 #5: 计算从下午3:00开始到相遇所需的时间
待完成 #6: 确定相遇的具体时间

波士顿到纽约的距离估计为215英里

待完成 #1: 估计波士顿到纽约之间的距离
待完成 #2: 计算第一列火车在下午2:00到3:00之间行驶的距离
待完成 #3: 计算第二列火车出发时两列火车之间的剩余距离
待完成 #4: 计算两列火车的相对速度（相向而行）
待完成 #5: 计算从下午3:00开始到相遇所需的时间
待完成 #6: 确定相遇的具体时间

第一列火车在1小时内行驶了60英里（60 mph × 1小时）

待完成 #1: 估计波士顿到纽约之间的距离
待完成 #2: 计算第一列火车在下午2:00到3:00之间行驶的距离
待完成 #3: 计算第二列火车出发时两列火车之间的剩余距离
待完成 #4: 计算两列火车的相对速度（相向而行）
待完成 #5: 计算从下午3:00开始到相遇所需的时间
待完成 #6: 确定相遇的具体时间

...

从下午3:00开始到相遇需要155/140小时，约等于1小时6.43分钟

待完成 #1: 估计波士顿到纽约之间的距离
待完成 #2: 计算第一列火车在下午2:00到3:00之间行驶的距离
待完成 #3: 计算第二列火车出发时两列火车之间的剩余距离
待完成 #4: 计算两列火车的相对速度（相向而行）
待完成 #5: 计算从下午3:00开始到相遇所需的时间
待完成 #6: 确定相遇的具体时间

相遇时间为下午4:06左右（更精确地说是下午4:06:26）

待完成 #1: 估计波士顿到纽约之间的距离
待完成 #2: 计算第一列火车在下午2:00到3:00之间行驶的距离
待完成 #3: 计算第二列火车出发时两列火车之间的剩余距离
待完成 #4: 计算两列火车的相对速度（相向而行）
待完成 #5: 计算从下午3:00开始到相遇所需的时间
待完成 #6: 确定相遇的具体时间

所有步骤已完成。现在让我提供最终解决方案：

解决方案

问题摘要：
- 第一列火车：下午2:00从波士顿出发，速度60英里/小时
- 第二列火车：下午3:00从纽约出发，速度80英里/小时，向波士顿方向行驶
- 波士顿到纽约的距离估计为 215英里

计算过程：

1. 第一步：第一列火车在下午2:00到3:00之间单独行驶了1小时
   - 行驶距离 = 60 × 1 = 60英里

2. 第二步：下午3:00时，两列火车之间的剩余距离
   - 剩余距离 = 215 - 60 = 155英里

3. 第三步：两列火车相向而行，相对速度
   - 相对速度 = 60 + 80 = 140英里/小时

4. 第四步：从下午3:00开始到相遇所需时间
   - 时间 = 155 ÷ 140 = 31/28 小时 ≈ 1.107小时
   - 转换为分钟：31/28 × 60 = 66又3/7 分钟 ≈ 66.43分钟
   - 即 1小时6分26秒

5. 第五步：相遇时间
   - 下午3:00 + 1小时6分26秒 = 下午4:06:26

最终答案：

两列火车将在 下午4:06左右（更精确地说是下午4:06:26）相遇。